# Collective modes of a minimal K2P gate model (2D, four rods, ten springs)
Code for: G. Frenkel, L. Mourokh, Y. Ben-Abu, "Collective Mechanical Modes in a Minimal Model of Ion Channel Gate Dynamics". Model units: length 1 nm, mass 1.44e-24 kg, spring constant 0.1 N/m (see the Table 1 cell for the conversion to SI).

### Requirements: see requirements.txt in the repository (numpy, scipy, matplotlib, sympy, pillow)

## Step 1: define the model and derive the symbolic Euler–Lagrange equations
1. Give the number of rods, springs and fixed points.
2. For each spring, give the two elements it connects (rods or fixed points).
3. Give the equilibrium state of every rod (x, y and tilt angle θ).
4. For every spring, give the attachment positions relative to the connected rods (p and q, in the rod's local frame).
5. Calculate the equilibrium length of every spring.
6. Derive the Euler–Lagrange equations.

Note: the total time derivative $\frac{d}{dt}\frac{\partial L}{\partial \dot q_i}$ is computed with the chain rule over all coordinates and velocities: $\sum_j \left[\frac{\partial^2 L}{\partial \dot q_i\,\partial q_j}\dot q_j + \frac{\partial^2 L}{\partial \dot q_i\,\partial \dot q_j}\ddot q_j\right]$.

In [ ]:
#--- Input Data:
num_rods = 4           # rods are numbered 1 2 3 ..
num_fixed_points = 4   # numbered 1,2,3,...
num_springs = 10        # numbered 1,2,3...
#--- spring connection key: r = rod , f=fixed_point
spring_connect = [["r1","r2"], ["r1","r2"], ["r1","r3"], ["r2","r4"], ["r1","r3"], ["r2","r4"],
                  ["r1","f2"], ["r3", "f1"], ["r2","f4"], ["r4", "f3"] ]
#--- fixed points (Fig. 1B): f1 = A1 left (-3, 2), f2 = A2 left (-3, -2), f3 = A1 right (3, 2), f4 = A2 right (3, -2)
#--- rod numbers vs location: 1 inner left, 2 inner right, 3 outer left, 4 outer right
import numpy as np
import sympy as sp

# Define symbolic variables for generalized coordinates and their derivatives
# Each rod has 3 degrees of freedom in 2D: x, y of its centre and the angle theta
q = sp.symbols([f'x{i} y{i} theta{i}' for i in range(1, num_rods + 1)])
dq = sp.symbols([f'dx{i} dy{i} dtheta{i}' for i in range(1, num_rods + 1)])
ddq = sp.symbols([f'ddx{i} ddy{i} ddtheta{i}' for i in range(1, num_rods + 1)])
fixed_points = sp.symbols([f'xf{i} yf{i}' for i in range(1, num_fixed_points + 1)])
q = [item for sublist in q for item in sublist]
dq = [item for sublist in dq for item in sublist]
ddq = [item for sublist in ddq for item in sublist]
fixed_points = [item for sublist in fixed_points for item in sublist]
print(q)
print(dq)
print(ddq)
print(fixed_points)

# Masses and moments of inertia
m_str = ""
for i in range(1, num_rods + 1):
    m_str = m_str + f'm{i} '
print(m_str)
m = sp.symbols(m_str) #sp.symbols('m1 m2 m3 m4')
I = [sp.symbols(f'I{i}zz') for i in range(1,num_rods +1)]    # [sp.symbols('I1zz'), sp.symbols('I2zz'), sp.symbols('I3zz')]
print("masses:", m)
print("I:", I)
# Spring constants and rest lengths
k_str = ""
l0_str = ""
for i in range(1, num_springs + 1):
    k_str = k_str + f'k{i} '
    l0_str = l0_str + f'l0{i} '
k = sp.symbols(k_str)      # sp.symbols('k1 k2 k3 k4 k5 k6')
l0 = sp.symbols(l0_str)    #sp.symbols('l01 l02 l03 l04 l05 l06')
print("k:", k)
print("l0:", l0)

# Define attachment points for the springs in the local frame of the rods
spring_local_pos_p = [sp.Matrix(sp.symbols(f'p{i}x p{i}y')) for i in range(1, num_springs+1)]
spring_local_pos_q = [sp.Matrix(sp.symbols(f'q{i}x q{i}y')) for i in range(1, num_springs+1)]

# 2D rotation matrix
def rotation_matrix(theta, isSmall=False):
    if isSmall == True:
        Rz = sp.Matrix([[1, -theta],
                    [theta,  1]])
        return Rz
    Rz = sp.Matrix([[sp.cos(theta), -sp.sin(theta)],
                    [sp.sin(theta),  sp.cos(theta)]])
    return Rz

# Kinetic Energy
T = 0

for i in range(num_rods):
    # Translational kinetic energy
    v = sp.Matrix([dq[3*i], dq[3*i+1]])
    T_trans = 0.5 * m[i] * v.dot(v)

    # Rotational kinetic energy
    omega = dq[3*i+2]
    T_rot = 0.5 * (I[i] * omega*omega)

    T += T_trans + T_rot

print("T\n",T)

# Potential Energy
U = 0
for j in range(num_springs):
    # the two ends of spring j (rod or fixed point)
    rod_a, rod_b = tuple(spring_connect[j])  
    print("tuple:" ,rod_a,rod_b)
    if rod_a[0] == "f":
        fixed_point_num = int(rod_a[1:]) - 1
        pos_a = sp.Matrix([fixed_points[2*(fixed_point_num)], fixed_points[2*(fixed_point_num)+1]])
    elif rod_a[0] == 'r':
        rod_num = int(rod_a[1:]) - 1
        R_a = rotation_matrix(q[3*rod_num+2])
        r_a = sp.Matrix([q[3*rod_num], q[3*rod_num+1]])
        pos_a = r_a + R_a * spring_local_pos_p[j]
    else:
        print("\n****************** ERROR - unknown spring connection ***************\n")
    if rod_b[0] == "f":
        fixed_point_num = int(rod_b[1:]) - 1
        pos_b = sp.Matrix([fixed_points[(2*fixed_point_num)], fixed_points[2*(fixed_point_num)+1]])
    elif rod_b[0] == "r":
        rod_num = int(rod_b[1:]) - 1
        R_b = rotation_matrix(q[3*rod_num+2])
        r_b = sp.Matrix([q[3*rod_num], q[3*rod_num+1]])
        pos_b = r_b + R_b * spring_local_pos_q[j]
    else:
        print("\n****************** ERROR - unknown spring connection ***************\n")

    dist = sp.sqrt((pos_a - pos_b).dot(pos_a - pos_b))
    U += 0.5 * k[j] * (dist - l0[j])**2

print("U\n", U)
    
# Lagrangian
L = T - U
print("The Lagrangian\n", L)

# Euler-Lagrange equations
EL_eqns = []
for i in range(len(q)):
    dL_dq = sp.diff(L, q[i])
    dL_ddq = sp.diff(L, dq[i])
    # total time derivative d/dt(dL/d(dq_i)): chain rule over all q_j and dq_j
    ddt_dL_ddq = 0
    for j, qj in enumerate(q):
        ddt_dL_ddq = ddt_dL_ddq + sp.diff(dL_ddq, qj) * dq[j]
    for j, dqj in enumerate(dq):
        ddt_dL_ddq = ddt_dL_ddq + sp.diff(dL_ddq, dqj) * ddq[j]
    EL_eqn = ddt_dL_ddq - dL_dq
    EL_eqns.append(EL_eqn)
    #EL_eqns.append(sp.simplify(EL_eqn))

# Display results
for i, eqn in enumerate(EL_eqns):
    print("----------------\n")
    print(f"Equation {i+1}: {eqn}")
    
    


In [ ]:
sp.init_printing(use_latex='mathjax')
L

## Step 2: numerical values and a plot of the system
1. Define a plotting function.
2. Give numerical values for all parameters.
3. Plot the system to check its structure (Fig. 1B).

In [ ]:
# plot rods springs and fixed points:
import numpy as np
import matplotlib.pyplot as plt

def plot_system(rod_data, spring_data, fixed_points_data):
    """
    Plots rods and springs in 2D.
    
    Parameters:
    - rod_data: List of tuples [(x, y, theta, length), ...]
    - spring_data: List of tuples [(x1, y1, x2, y2), ...]
    """
    fig, ax = plt.subplots(figsize=(8, 6))
    
    # Plot rods
    for x, y, theta, length in rod_data:
        dx = (length / 2) * np.cos(theta)
        dy = (length / 2) * np.sin(theta)
        ax.plot([x - dx, x + dx], [y - dy, y + dy], 'b-', linewidth=8)

    # Plot springs
    for x1, y1, x2, y2 in spring_data:
        ax.plot([x1, x2], [y1, y2], 'r--', linewidth=2)  # Basic spring as dashed line

    # Plot fixed points as circle:
    for x,y in fixed_points_data:
        ax.plot(x,y,"ro", markersize=20)
    
    ax.set_aspect('equal')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    plt.grid(True)
    #plt.show()
    return fig, ax   # return them so you can add to the plot later


## System data
### Input (model units)
1. rod_data: equilibrium position (x, y) and orientation θ of each rod, and its (constant) length.
2. spring_connect: the two elements each spring connects, (rod_n, rod_m) or (rod_n, fixed point_m).
3. spring_local_distance: position of each attachment point along the rod axis, measured from the rod centre (positive towards the upper end of the rod, i.e. in the direction θ); 0 for fixed points.
4. fixed_points_data: coordinates (x, y) of the fixed points.
5. spring_constants
6. rod_masses
7. InertiaZZ: moments of inertia of uniform rods, I = mL²/12.
### Calculated
8. springs_rest_length

All values are in model units (length nm, mass 1.44e-24 kg, spring constant 0.1 N/m). The rest length of every spring is set equal to its length at equilibrium (next cell), so the network is unstressed; this is why the model has exactly two zero-frequency modes.

In [ ]:
import numpy as np

def sub_numeric_constants(sympy_eq, spring_local_distance, rod_data, spring_connect, fixed_points_data,
                          spring_constants, springs_rest_length, rod_masses, inertia_zz):
    """
    Substitutes numerical constants into a symbolic SymPy equation.

    Parameters:
        sympy_eq (sympy.Expr): The symbolic equation.
        spring_local_distance (list): Local distances of springs relative to the center of mass of the relevant rods.
        rod_data (list): Rod position and angle and length, data.
        spring_connect (list): Connections between rods and springs for each spring the names of the connected rods/fixed point pair are given.
        fixed_points_data (list): Coordinates of fixed points.
        spring_constants (list): Spring stiffness values.
        springs_rest_length (list): Rest lengths of the springs.
        rod_masses (list): Mass values for rods.
        inertia_zz (list): Moments of inertia for rods.

    Returns:
        sympy.Expr: The equation with numerical values substituted.
    """
    numerical_eq = sympy_eq

    # Substitute spring connection vectors relative to rod centers
    for idx, connection in enumerate(spring_connect):
        for point_idx, spring_point in enumerate(connection):
            if spring_point.startswith("r"):
                rod_index = int(spring_point[1:]) - 1
                local_x = spring_local_distance[idx][point_idx] #spring_local_distance[idx][point_idx] * np.cos(rod_data[rod_index][2])
                local_y = 0                                     #spring_local_distance[idx][point_idx] * np.sin(rod_data[rod_index][2])

                target_var = spring_local_pos_p if point_idx == 0 else spring_local_pos_q
                numerical_eq = numerical_eq.subs({target_var[idx][0]: local_x, target_var[idx][1]: local_y})

    # Substitute fixed points
    """
    numerical_eq = numerical_eq.subs({
        fixed_points[2 * i]: fixed_points_data[i][0],
        fixed_points[2 * i + 1]: fixed_points_data[i][1]
    } for i in range(len(fixed_points_data)))
    """
    for i in range(len(fixed_points_data)):
        #print("i=",i)
        numerical_eq = numerical_eq.subs({
            fixed_points[2 * i]: fixed_points_data[i][0],
            fixed_points[2 * i + 1]: fixed_points_data[i][1]
        })

    
    # Substitute spring constants, rest lengths, rod masses, and moments of inertia
    for symbol_list, values in zip([k, l0, m, I], [spring_constants, springs_rest_length, rod_masses, inertia_zz]):
        numerical_eq = numerical_eq.subs({sym: val for sym, val in zip(symbol_list, values)})

    return numerical_eq


In [ ]:
def get_numerical_eq(sympy_eq, rod_data):
    """Substitutes symbolic variables in sympy_eq with numerical values from rod_data."""
    numerical_eq = sympy_eq

    for indx, qrod in enumerate(q):
        rod_index = indx // 3  # Determine which rod
        coord_index = indx % 3  # Determine which coordinate (x, y, or theta)
        numerical_eq = numerical_eq.subs(qrod, rod_data[rod_index][coord_index])

    return numerical_eq

In [ ]:
# Model parameters (model units: nm, 1.44e-24 kg, 0.1 N/m)


rod_data = [(-1, 0, np.pi/2 - np.pi/9, 3), (1, 0, np.pi/2 + np.pi/9, 3), (-1.1, 0.5, np.pi/2 -np.pi/20, 2),
           (1.1, 0.5, np.pi/2 + np.pi/20, 2)]  # (x, y, theta, length)

spring_connect = [["r1","r2"], ["r1","r2"], ["r1","r3"], ["r2","r4"], ["r1","r3"], ["r2","r4"],
                  ["r1","f2"], ["r3", "f1"], ["r2","f4"], ["r4", "f3"] ]

spring_local_distance = [[1,1],[-1,-1], [-0.5,-1.0], [-0.5,-1.0], [1.4,0.85], [1.4,0.85],
                        [-1.5,0], [1,0], [-1.5,0], [1,0]]

fixed_points_data = [(-3,2), (-3,-2), (3,2), (3,-2)]

spring_constants = [10,20, 1000,1000, 10,10, 10,10,10,10]
# model units of 0.1 N/m. i.e. = 1, 2, 100 N/m in the physical model - translated by k_unit below. 

rod_masses = [3,3,2,2]
# model units of 1.44e-24 kg, i.e. 4.32e-24 and 2.88e-24 kg (2.6, 1.73 kDa)

InertiaZZ = [rod_masses[n]*(rod_data[n][3]**2)/12.0 for n in range(len(rod_data))]

springs_rest_length = [2,3, 1,1, 0.1,0.1, 2,2,2,2]
# placeholders, replaced in the next cell by the equilibrium lengths

#--- units conversion from model units to physical units
k_unit, m_unit = 0.1, 1.44e-24          # N/m and kg per model unit (used for SI output in the Table 1 cell)
print("spring constants (N/m):", [k * k_unit for k in spring_constants])
print("masses (kg):", [mm * m_unit for mm in rod_masses])

In [ ]:
# Calculating the rest length of the springs
#-------------------------------------------
for j in range(num_springs):
    # the two ends of spring j (rod or fixed point)
    rod_a, rod_b = tuple(spring_connect[j])  
    print("tuple:" ,rod_a,rod_b)
    if rod_a[0] == "f":
        fixed_point_num = int(rod_a[1:]) - 1
        pos_a = sp.Matrix([fixed_points[2*(fixed_point_num)], fixed_points[2*(fixed_point_num)+1]])
    elif rod_a[0] == 'r':
        rod_num = int(rod_a[1:]) - 1
        R_a = rotation_matrix(q[3*rod_num+2])
        r_a = sp.Matrix([q[3*rod_num], q[3*rod_num+1]])
        pos_a = r_a + R_a * spring_local_pos_p[j]
    else:
        print("\n****************** ERROR - unknown spring connection ***************\n")
    
    if rod_b[0] == "f":
        fixed_point_num = int(rod_b[1:]) - 1
        pos_b = sp.Matrix([fixed_points[(2*fixed_point_num)], fixed_points[2*(fixed_point_num)+1]])
        
    elif rod_b[0] == "r":
        rod_num = int(rod_b[1:]) - 1
        R_b = rotation_matrix(q[3*rod_num+2])
        r_b = sp.Matrix([q[3*rod_num], q[3*rod_num+1]])
        pos_b = r_b + R_b * spring_local_pos_q[j]
    else:
        print("\n****************** ERROR - unknown spring connection ***************\n")

    dist = sp.sqrt((pos_a - pos_b).dot(pos_a - pos_b))
    dist = sub_numeric_constants(dist, spring_local_distance, rod_data, spring_connect, fixed_points_data,
                                    spring_constants,[], rod_masses, InertiaZZ)
    dist = get_numerical_eq(dist,rod_data)
    print(dist)
    springs_rest_length[j] = dist

In [ ]:
#--- plotting the system
spring_data = []
indx = 0
for s in spring_connect:
    if s[0][0] == "r":
        rod_num = int(s[0][1:]) - 1
        spring_x1 = rod_data[rod_num][0] + spring_local_distance[indx][0]*np.cos(rod_data[rod_num][2])
        spring_y1 = rod_data[rod_num][1] + spring_local_distance[indx][0]*np.sin(rod_data[rod_num][2])
    else:
        fixed_point_num = int(s[0][1:]) - 1
        spring_x1 = fixed_points_data[fixed_point_num][0]
        spring_y1 = fixed_points_data[fixed_point_num][1]
    if s[1][0] == "r":
        rod_num = int(s[1][1:]) - 1
        spring_x2 = rod_data[rod_num][0]  +  spring_local_distance[indx][1]*np.cos(rod_data[rod_num][2])
        spring_y2 = rod_data[rod_num][1]  +  spring_local_distance[indx][1]*np.sin(rod_data[rod_num][2])
    else:
        fixed_point_num = int(s[1][1:]) - 1
        spring_x2 = fixed_points_data[fixed_point_num][0]
        spring_y2 = fixed_points_data[fixed_point_num][1]
        
    print(spring_x1, spring_y1, spring_x2, spring_y2)
    spring_data.append((spring_x1, spring_y1, spring_x2, spring_y2))
    indx+=1
plot_system(rod_data, spring_data, fixed_points_data)

In [ ]:
# Call the function
fig, ax = plot_system(rod_data, spring_data, fixed_points_data)

# Now add text outside the function
ax.text(-2.9, 1.7, "$A_1$", fontsize=14, color="black")
ax.text(2.7, 1.7, "$A_1$", fontsize=14, color="black")
ax.text(-3, -1.8, "$A_2$", fontsize=14, color="black")
ax.text(2.7, -1.8, "$A_2$", fontsize=14, color="black")
ax.text(0, 1.1, "$k_1$", fontsize=14, color="black")
ax.text(0, -0.8, "$k_2$", fontsize=14, color="black")
ax.text(-2, 1.8, "$k_3$", fontsize=14, color="black")
ax.text(1.78, 1.8, "$k_4$", fontsize=14, color="black")
ax.text(1.78, -1.8, "$k_6$", fontsize=14, color="black")
ax.text(-2, -1.8, "$k_5$", fontsize=14, color="black")
ax.text(0.6, 1.4, "$k_8$", fontsize=14, color="black")
ax.text(-0.8, 1.4, "$k_7$", fontsize=14, color="black")
ax.text(1.35, -0.7, "$k_{10}$", fontsize=14, color="black")
ax.text(-1.55, -0.7, "$k_{9}$", fontsize=14, color="black")
ax.text(-0.8, 0.0, "Rod 1", fontsize=14, color="black")
ax.text(0.3, 0.0, "Rod 2", fontsize=14, color="black")
ax.text(-1.9, 0.0, "Rod 3", fontsize=14, color="black")
ax.text(1.3, 0.0, "Rod 4", fontsize=14, color="black")
#fig.savefig("figure 1b.pdf", bbox_inches="tight")
#fig.savefig("figure 1b 600 dpi.png", dpi=600, bbox_inches="tight")
fig.savefig("figure_1b.svg", bbox_inches="tight", pad_inches=0)
plt.show()

### Check: at equilibrium all generalised forces vanish (only the inertial terms remain)

In [ ]:
#--- Test: Getting Numeric values for the accelerations
nnn=[]
for indx in range(num_rods*3):
    nnn.append(sub_numeric_constants(EL_eqns[indx], spring_local_distance, rod_data, spring_connect, fixed_points_data,
                                     spring_constants,springs_rest_length, rod_masses, InertiaZZ))
    #print(nnn[indx])
for eq in nnn:
    print(get_numerical_eq(eq, rod_data))

## Linearise the equations
Near equilibrium each Euler–Lagrange equation has the form $C_i\,\ddot q_i + (\text{linear terms in the } q_j) = 0$, with $C_i = m$ for a translation and $C_i = I$ for a rotation. From here on $q$ denotes the deviation from equilibrium, $q \equiv \Delta q = q - q_{eq}$.

In this form the normal modes follow from a generalised eigenvalue problem. The mass-weighted coordinates $\tilde q_i = \sqrt{C_i}\,q_i$ (next cells) turn it into a standard symmetric one. There are 3 × 4 = 12 equations.

## Eigenvalues and eigenvectors (Goldstein, Classical Mechanics, Ch. 6)
Mass-weighted coordinates:
- $x \to \tilde{x} = x\sqrt{m}$, so that $\tfrac12 m\dot{x}^2 = \tfrac12\dot{\tilde{x}}^2$;
- $\theta \to \tilde{\theta} = \theta\sqrt{I}$, so that $\tfrac12 I\dot{\theta}^2 = \tfrac12\dot{\tilde{\theta}}^2$.

Both have units of length·√mass, so they can be combined in one eigenvector.

Steps:
1. Substitute the new coordinates into the Lagrangian.
2. Derive the Euler–Lagrange equations again for the new coordinates.
3. Linearise them: $\ddot{\tilde q} = -K\tilde q$, with K symmetric. Its eigenvalues are $\omega_n^2$ (model units). K = −C̃ in the paper's notation.

In [ ]:
# Change coordinates to the mass weighted coordinates by:  L(q,\dot{q})=L(\tilde{q})/\sqrt(C_i},\dot{\tilde{q})}/\sqrt(C_i})
L_factored = L
for i,qi in enumerate(q):
    rod_num_minus1 = i//3  
    if (i+1)%3 == 0:
        L_factored = L_factored.subs({qi: qi/sp.sqrt(I[rod_num_minus1])})
    else:
        L_factored = L_factored.subs({qi: qi/sp.sqrt(m[rod_num_minus1])})
for i,dqi in enumerate(dq):
    rod_num_minus1 = i//3  
    if (i+1)%3 == 0:
        L_factored = L_factored.subs({dqi: dqi/sp.sqrt(I[rod_num_minus1])})
    else:
        L_factored = L_factored.subs({dqi: dqi/sp.sqrt(m[rod_num_minus1])})

L_factored

In [ ]:
# Euler-Lagrange equations
EL_eqns = []
for i in range(len(q)):
    dL_dq = sp.diff(L_factored, q[i])
    dL_ddq = sp.diff(L_factored, dq[i])
    ddt_dL_ddq = 0
    for j, qj in enumerate(q):
        ddt_dL_ddq = ddt_dL_ddq + sp.diff(dL_ddq, qj) * dq[j]
    for j, dqj in enumerate(dq):
        ddt_dL_ddq = ddt_dL_ddq + sp.diff(dL_ddq, dqj) * ddq[j]
    EL_eqn = ddt_dL_ddq - dL_dq
    EL_eqns.append(EL_eqn)
    #EL_eqns.append(sp.simplify(EL_eqn))

# Display results
for i, eqn in enumerate(EL_eqns):
    print("----------------\n")
    sp.pprint(f"Equation {i+1}: {eqn}")

In [ ]:
#------------- refactor rod data due to coordinate change ---------------------
rod_data_fact = list(np.zeros([len(rod_data),4]))
for i in range(len(rod_data)):
    rod_data_fact[i][0] = rod_data[i][0]*np.sqrt(rod_masses[i])
    rod_data_fact[i][1] = rod_data[i][1]*np.sqrt(rod_masses[i])
    rod_data_fact[i][2] = rod_data[i][2]*np.sqrt(InertiaZZ[i])
    rod_data_fact[i][3] = rod_data[i][3]    #The length of rods is constant and wasn't calibrated 
sp.pprint(rod_data_fact)

In [ ]:
#--- Test: Getting Numeric values for the accelerations
#--- Check: at equilibrium the spring forces should vanish. The small residuals (~1e-4) are numerical and do not affect the linearised coefficients (verified against scripts/check_model.py).
nnn=[]
for indx in range(num_rods*3):
    nnn.append(sub_numeric_constants(EL_eqns[indx], spring_local_distance, rod_data_fact, spring_connect, fixed_points_data,
                                    spring_constants,springs_rest_length, rod_masses, InertiaZZ))
    #print(nnn[indx])
for eq in nnn:
    print(sp.N(get_numerical_eq(eq, rod_data_fact),5))

In [ ]:
#--- Get the linear Euler lagrange equation in the mass weighted coordinates
linearEqs = []
for eq_num in range(3*len(rod_data)):
    linearEq = 0
    eq = sub_numeric_constants(EL_eqns[eq_num], spring_local_distance, rod_data_fact, spring_connect, fixed_points_data,
                                  spring_constants,springs_rest_length, rod_masses, InertiaZZ )
    for indx in range(len(q)):
        #print(q[indx])
        tempEq = sp.diff(eq, q[indx])
        tempEq = get_numerical_eq(tempEq, rod_data_fact)
        linearEq = linearEq + tempEq*q[indx]
    print(sp.N(linearEq,15))
    #--- divide the equation by the prefactor of the second derivative
    print("    second derivative prefactor = ", sp.diff(eq, ddq[eq_num]))
    linearEqs.append(sp.N(linearEq,15))#/sp.diff(eq, ddq[eq_num]))
linearEqs
sp.pretty_print(linearEqs)

In [ ]:
#--- check the equation more comfortably with a lower number of digits 
sp.pprint(sp.N(linearEqs[0],5))

### The coefficient matrix K: $\ddot{\tilde{q}} = -K\tilde{q}$. K is symmetric; its eigenvalues are $\omega_n^2$ (model units) and its eigenvectors are the normal modes.

In [ ]:
# Getting the coefficients matrix eigen values and vectors ---
coef_mat = np.zeros([3*len(rod_data),3*len(rod_data)])
for i, qi in enumerate(q):
    coeff_dict = linearEqs[i].as_coefficients_dict()
    for j, qj in enumerate(q):
        coef_mat[i][j] = coeff_dict.get(qj, 0)
# Compute eigenvalues and eigenvectors (normal modes)
K = sp.Matrix(coef_mat)
sp.pprint(sp.N(K,3))
eigenvals, eigenvecs = K.eigenvals(), K.eigenvects()

print("\nEigenvalues (ω², model units):", eigenvals)
print("\nEigenvectors (Normal Modes):")
for eig in eigenvecs:
    print(f"Eigenvalue: {eig[0]}, Mode shape: {eig[2]}")


## Motion of each normal mode over one period (Fig. 2, Fig. 3, animations)
Set index_mode to the SymPy index of the mode (see the Table 1 cell for the paper mode number).

In [ ]:
def flatten(xss):
    return [x for xs in xss for x in xs]

In [ ]:
# Paper mode numbers (ascending frequency; modes 1-2 are the zero modes): sympy index -> paper number
# dictionary: old_number -> new_number
mode_map = {
    2: 1,
    3: 2,
    1: 3,
    0: 4,
    4: 5,
    5: 6,
    6: 7,
    8: 8,
    9: 9,
    7: 10,
    11: 11,
    10: 12
}

# function to map
def remap_mode(old_number):
    return mode_map.get(old_number, None)  # None if not found

In [ ]:
# --- Table 1: convert the eigenvalues (model units) to SI angular frequencies ---
# Model units (set in the model-parameters cell): length 1 nm, mass m_unit = 1.44e-24 kg,
# spring constant k_unit = 0.1 N/m, i.e. the model values 10, 20, 1000 are translated to physical 1, 2, 100 N/m.
# The eigenvalues of coef_mat are omega^2 in units of f_k/f_m; lengths cancel.
f_k, f_m = k_unit, m_unit
omega_unit = np.sqrt(f_k / f_m)              # 2.635e11 rad/s; model time unit = 1/omega_unit = 3.79 ps
lam = np.array([float(e[0]) for e in eigenvecs])
order = np.argsort(lam)                      # paper mode n = n-th smallest eigenvalue
print(f"{'mode':>4} {'sympy index':>11} {'omega^2 (model units)':>22} {'omega (rad/s)':>14} {'period (ps)':>12}")
for n, old in enumerate(order, start=1):
    if n <= 2:
        print(f"{n:>4} {old:>11} {lam[old]:>22.3e} {'0 (zero mode)':>14} {'-':>12}")
        continue
    w = np.sqrt(lam[old]) * omega_unit
    print(f"{n:>4} {old:>11} {lam[old]:>22.6g} {w:>14.3e} {2*np.pi/w*1e12:>12.3g}")
    assert mode_map[old] == n, "mode_map does not follow the ascending order of the eigenvalues"

In [ ]:
#--- The time evolution of modes in one period starting at the maximal amplitude ---
#----------- solved using theoretical harmonic motion to find A and \phi ------------
index_mode = 10 # SymPy index; 10 = paper mode 12 (see the Table 1 cell)
new_mode_number = remap_mode(index_mode)

num_times = 100
resultsAll = []
omega_for_mode = np.sqrt(float(eigenvecs[index_mode][0]))
dt = ((2*np.pi) / omega_for_mode )/float(num_times-1)
t_list = list(range(num_times))
t_list = [float(x) * dt for x in t_list]

for component in range(12):
    results1 = []
    for plottime in range(num_times):
        results1.append( eigenvecs[index_mode][2][0][component]*np.cos(plottime*dt*omega_for_mode) )
    resultsAll.append(results1) 
#for component in range(12):
#    plt.plot(t_list,resultsAll[component])
#plt.plot(resultsAll[0])
#plt.plot(resultsAll[3])
#plt.show()
omega_for_indx= np.sqrt(float(eigenvecs[index_mode][0]))
period_T_for_indx = 2*np.pi/np.sqrt(float(eigenvecs[index_mode][0]))
t_list_normalized_by_period = [x/period_T_for_indx for x in t_list]

colors = ['tab:blue','tab:orange','tab:green','tab:red','tab:purple','tab:brown',
          'tab:pink','tab:gray','tab:olive','tab:cyan','black','gold']
linestyles = ['--','-.','-','--','-.','-','--','-.','-','--','-.','-']

comp_names = ['x', 'y', r'\theta']  # note r'\theta'

for comp in range(12):
    plt.plot(t_list_normalized_by_period, resultsAll[comp], 
             linestyle=linestyles[comp], color=colors[comp],
             label=r"$\tilde{" + comp_names[comp % 3] + "}" + f"{comp//3+1}(t)$")

plt.rcParams.update({
    "font.size": 16,           # default text size
    "axes.labelsize": 16,      # x and y labels
    "axes.titlesize": 20,      # title
    "xtick.labelsize": 14,     # x tick labels
    "ytick.labelsize": 14,     # y tick labels
    "legend.fontsize": 12      # legend
})
plt.xlabel(r"$t/T$")
#plt.ylabel("General Coordinates")
#plt.legend()
#plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))  # Outside right
plt.grid()

plt.text(0.065, 0.9, f"({new_mode_number})", transform=plt.gca().transAxes,
             fontsize=24, verticalalignment='top', horizontalalignment='left',
        bbox=dict(facecolor='white', alpha=0.6, edgecolor='black', boxstyle='round'))

#plt.savefig('Time Evolution of coordinates - new mode ' + str(new_mode_number) +'.png', bbox_inches='tight')
plt.savefig('Time Evolution of coordinates - new mode ' + str(new_mode_number) +'.svg', bbox_inches="tight", pad_inches=0)
plt.show()


## Fig. 3: change of the channel width Δx (dashed) and of the asymmetric shift Δy (dot-dashed) at the top, centre and bottom of the inner rods, over one period. Δ = right rod minus left rod (Eqs. 8–9).

In [ ]:
# Simulated motion
N = num_rods
timesteps = len(resultsAll[0])
x_cm   = np.zeros([N,timesteps])
y_cm   = np.zeros([N,timesteps])
angles = np.zeros([N,timesteps])
for i in range(N):
    # convert mass-weighted to physical displacements: x = x~/sqrt(m), theta = theta~/sqrt(I); 0.2 = display amplitude
    x_cm[i, :]   = resultsAll[i*3]/np.sqrt(rod_masses[i])*0.2
    y_cm[i, :]   = resultsAll[(i*3)+1]/np.sqrt(rod_masses[i])*0.2

    angles[i, :] = resultsAll[(i*3)+2]/np.sqrt(InertiaZZ[i])*0.2

In [ ]:
def get_rod_edge(rod_num, top_or_bottom, rod_data, x_cm,y_cm,angles):
    x=0.0
    y=0.0
    if top_or_bottom == "bottom":
        #--- bottom edge of rod i:
        x = rod_data[rod_num][0] + x_cm[rod_num, :] - (rod_data[rod_num][3] / 2) * np.cos(angles[rod_num,:]+rod_data[rod_num][2])
        y = rod_data[rod_num][1] + y_cm[rod_num, :] - (rod_data[rod_num][3] / 2) * np.sin(angles[rod_num,:]+rod_data[rod_num][2])
        #x = rod_data[rod_num][0] + np.array(resultsAll[rod_num*3]) - (rod_data[rod_num][3] / 2) * np.cos(angles[rod_num,:]+rod_data[i][2])
        #y = rod_data[rod_num][1] + resultsAll[rod_num*3+1] - (rod_data[rod_num][3] / 2) * np.sin(angles[rod_num,:]+rod_data[i][2])
    elif top_or_bottom == "top":
        #--- top edge of rod
        x = rod_data[rod_num][0] + x_cm[rod_num, :] + (rod_data[rod_num][3] / 2) * np.cos(angles[rod_num,:]+rod_data[rod_num][2])
        y = rod_data[rod_num][1] + y_cm[rod_num, :] + (rod_data[rod_num][3] / 2) * np.sin(angles[rod_num,:]+rod_data[rod_num][2])
    elif top_or_bottom == "center":
        #--- center of rod
        x = rod_data[rod_num][0] + x_cm[rod_num, :] 
        y = rod_data[rod_num][1] + y_cm[rod_num, :]
    else: #return the center_point
        print("************ ERROR: choose top bottom or center ************")
        x=0
        y=0
    return (x,y)

In [ ]:
#---equilibrium distances:
# center
D01_center_eq = rod_data[1][0]-rod_data[0][0]

# top
x_top_0_eq = rod_data[0][0] + (rod_data[0][3] / 2) * np.cos(rod_data[0][2])
x_top_1_eq = rod_data[1][0] + (rod_data[1][3] / 2) * np.cos(rod_data[1][2])
D01_top = x_top_1_eq-x_top_0_eq

# bottom
x_bottom_0_eq = rod_data[0][0]- (rod_data[0][3] / 2) * np.cos(rod_data[0][2])
x_bottom_1_eq = rod_data[1][0]- (rod_data[1][3] / 2) * np.cos(rod_data[1][2])
D01_bottom = x_bottom_1_eq-x_bottom_0_eq

print("x distance rods 0 1 at top = " + str(D01_top) + "  at bottom = " + str(D01_bottom) + " at center = " + str(D01_center_eq) )

#--- time dependent distances compared to equilibrium
D01_center_t = ((get_rod_edge(1,"center",rod_data,x_cm,y_cm,angles)[0]))-((get_rod_edge(0,"center",rod_data,x_cm,y_cm,angles)[0]))-D01_center_eq
D01_top_t = ((get_rod_edge(1,"top",rod_data,x_cm,y_cm,angles)[0]))-((get_rod_edge(0,"top",rod_data,x_cm,y_cm,angles)[0]))-D01_top
D01_bottom_t = ((get_rod_edge(1,"bottom",rod_data,x_cm,y_cm,angles)[0]))-((get_rod_edge(0,"bottom",rod_data,x_cm,y_cm,angles)[0]))-D01_bottom

#--- time dependent distances compared to equilibrium
y_D01_center_t = ((get_rod_edge(1,"center",rod_data,x_cm,y_cm,angles)[1]))-((get_rod_edge(0,"center",rod_data,x_cm,y_cm,angles)[1]))
y_D01_top_t = ((get_rod_edge(1,"top",rod_data,x_cm,y_cm,angles)[1]))-((get_rod_edge(0,"top",rod_data,x_cm,y_cm,angles)[1]))
y_D01_bottom_t = ((get_rod_edge(1,"bottom",rod_data,x_cm,y_cm,angles)[1]))-((get_rod_edge(0,"bottom",rod_data,x_cm,y_cm,angles)[1]))

In [ ]:
# --- fixed color and linestyle mapping ---
colors = {
    "top_x": "tab:blue",
    "bottom_x": "tab:orange",
    "center_x": "tab:green",
    "top_y": "tab:red",
    "bottom_y": "tab:purple",
    "center_y": "tab:brown",
}
linestyles = {
    "top_x": "--",
    "bottom_x": "--",
    "center_x": "--",
    "top_y": "-.",
    "bottom_y": "-.",
    "center_y": "-.",
}

# --- data mapping for looping ---
curves = {
    "top_x": (D01_top_t, r"$\Delta x_{top}$"),
    "bottom_x": (D01_bottom_t, r"$\Delta x_{bottom}$"),
    "center_x": (D01_center_t, r"$\Delta x_{center}$"),
    "top_y": (y_D01_top_t, r"$\Delta y_{top}$"),
    "bottom_y": (y_D01_bottom_t, r"$\Delta y_{bottom}$"),
    "center_y": (y_D01_center_t, r"$\Delta y_{center}$"),
}

# --- plotting ---
#plt.figure(figsize=(8,6))
plt.rcParams.update({
    "font.size": 16,           # default text size
    "axes.labelsize": 16,      # x and y labels
    "axes.titlesize": 20,      # title
    "xtick.labelsize": 14,     # x tick labels
    "ytick.labelsize": 14,     # y tick labels
    "legend.fontsize": 12      # legend
})

for key, (data, label) in curves.items():
    plt.plot(t_list_normalized_by_period, data,
             color=colors[key], linestyle=linestyles[key], label=label)

# mode number annotation
plt.text(0.065, 0.9, f"({new_mode_number})", transform=plt.gca().transAxes,
         fontsize=24, verticalalignment='top', horizontalalignment='left',
         bbox=dict(facecolor='white', alpha=0.6, edgecolor='black', boxstyle='round'))

plt.xlabel(r"$t/T$")
#plt.ylabel("Distance − Equilibrium Distance")
#plt.ylabel(r"$\Delta$")
#plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))  # outside right
plt.grid(True)  # add gridlines

#plt.savefig(f'Figure 3 new_mode {new_mode_number}.png', bbox_inches='tight')
plt.savefig(f'Figure 3 new_mode {new_mode_number}.svg', bbox_inches='tight')
plt.show()


# Animation of rod motion

In [ ]:
#----------------- find the end point of each line at each time frame and create for each rod animation
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

N = len(rod_data)
timesteps = len(resultsAll[0])
print("N=",N, "timesteps=", timesteps)

# Simulated motion
x_cm   = np.zeros([N,timesteps])
y_cm   = np.zeros([N,timesteps])
angles = np.zeros([N,timesteps])
for i in range(N):
    # convert mass-weighted to physical displacements: x = x~/sqrt(m), theta = theta~/sqrt(I); 0.2 = display amplitude
    x_cm[i, :]   = resultsAll[3*i]/np.sqrt(rod_masses[i])*0.2
    y_cm[i, :]   = resultsAll[3*i+1]/np.sqrt(rod_masses[i])*0.2
    
    angles[i, :] = resultsAll[3*i+2]/np.sqrt(InertiaZZ[i])*0.2


# Initialize figure
fig, ax = plt.subplots()
ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)

# mode number annotation
ax.text(0.065, 0.9, f"({new_mode_number})", transform=plt.gca().transAxes,
         fontsize=24, verticalalignment='top', horizontalalignment='left',
         bbox=dict(facecolor='white', alpha=0.6, edgecolor='black', boxstyle='round'))
ax.grid(True)  # add gridlines

# Create line objects for each rod
lines = [ax.plot([], [], 'o-', lw=2)[0] for _ in range(N)]

def init():
    """Initialize the animation."""
    for line in lines:
        line.set_data([], [])
    return lines

def update(frame):
    """Update function for animation."""
    for i in range(N):
        # Compute endpoints of each rod
        x1 = rod_data[i][0] + x_cm[i, frame] - (rod_data[i][3] / 2.0) * np.cos(angles[i,frame]+rod_data[i][2])
        y1 = rod_data[i][1] + y_cm[i, frame] - (rod_data[i][3] / 2.0) * np.sin(angles[i,frame]+rod_data[i][2])
        x2 = rod_data[i][0] + x_cm[i, frame] + (rod_data[i][3] / 2.0) * np.cos(angles[i,frame]+rod_data[i][2])
        y2 = rod_data[i][1] + y_cm[i, frame] + (rod_data[i][3] / 2.0) * np.sin(angles[i,frame]+rod_data[i][2])
        
        # Update line data
        lines[i].set_data([x1, x2], [y1, y2])
    
    return lines

# Create animation
ani = animation.FuncAnimation(fig, update, frames=timesteps, init_func=init, blit=True, interval=20)
#plt.show()
from IPython.display import HTML

# Display animation in Jupyter Notebook
HTML(ani.to_jshtml())

In [ ]:
ani.save(f"rods_animation_mode_{new_mode_number}_one_period.gif", writer="pillow")

# --- END of the main code ---